In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from glob import glob
from tqdm.auto import tqdm
import datetime

from matplotlib.patches import Ellipse
import matplotlib.pyplot as plt

In [2]:
def split_dataset(X, Y, train_ratio=0.8, train_sample_ratio=1.0, eval_sample_ratio=1.0, seed=42):
    total_len = len(X)
    train_len = int(total_len * train_ratio)

    X_train_full, Y_train_full = X[:train_len], Y[:train_len]
    X_eval_full, Y_eval_full = X[train_len:], Y[train_len:]

    rng = np.random.default_rng(seed)

    if train_sample_ratio < 1.0:
        indices = rng.choice(len(X_train_full), int(len(X_train_full) * train_sample_ratio), replace=False)
        X_train, Y_train = X_train_full[indices], Y_train_full[indices]
    else:
        X_train, Y_train = X_train_full, Y_train_full

    if eval_sample_ratio < 1.0:
        indices = rng.choice(len(X_eval_full), int(len(X_eval_full) * eval_sample_ratio), replace=False)
        X_eval, Y_eval = X_eval_full[indices], Y_eval_full[indices]
    else:
        X_eval, Y_eval = X_eval_full, Y_eval_full

    return MDNDataset(X_train, Y_train), MDNDataset(X_eval, Y_eval)

def get_dataloader(dataset, batch_size=512, shuffle=True):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0)

def visualize_mdn_output(ax, model, input_tensor, device):
    input_tensor = tuple(t.view(1, -1).to(device) for t in input_tensor)
    model.eval()
    with torch.no_grad():
        pi, mu, sigma = model(input_tensor)
        pi = pi.squeeze().cpu().numpy()
        mu = mu.squeeze().cpu().numpy()
        sigma = sigma.squeeze().cpu().numpy()

    x = np.linspace(-2, 2, 200)
    y = np.linspace(-2, 2, 200)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)

    for k in range(len(pi)):
        mux, muy = mu[k]
        sigx, sigy = sigma[k]
        px = (1.0 / (np.sqrt(2 * np.pi) * sigx)) * np.exp(-0.5 * ((X - mux) / sigx) ** 2)
        py = (1.0 / (np.sqrt(2 * np.pi) * sigy)) * np.exp(-0.5 * ((Y - muy) / sigy) ** 2)
        Z += pi[k] * px * py

    ax.contourf(X, Y, Z, levels=100, cmap="viridis")
    ax.scatter(mu[:, 0], mu[:, 1], c='red', s=40, edgecolors='white', label='mu')
    ax.set_title("GMM")
    ax.set_xlabel("delta_x")
    ax.set_ylabel("delta_y")
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.grid(True)
    ax.legend()

# --------------------- 데이터 로딩 ---------------------
def load_dataset(data_dir, pattern="mdn_data_*.npz"):
    npz_files = sorted(glob(os.path.join(data_dir, pattern)))
    input_list, target_list = [], []

    for file in npz_files:
        data = np.load(file)
        input_list.append(data['input'])
        target_list.append(data['target'])

    X = np.concatenate(input_list, axis=0)
    Y = np.concatenate(target_list, axis=0)
    return X, Y

# --------------------- Dataset 정의 ---------------------
class MDNDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y
        self.t_scale = 2400.0
        self.coord_scale = 100000.0

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        t, x, y = self.X[idx]
        x = torch.tensor([x / self.coord_scale], dtype=torch.float32)
        y = torch.tensor([y / self.coord_scale], dtype=torch.float32)
        t = torch.tensor([t / self.t_scale], dtype=torch.float32)
        return (x, y, t), torch.from_numpy(self.Y[idx]).float() / self.coord_scale

# --------------------- MDN 모델 정의 ---------------------
class MDN(nn.Module):
    def __init__(self, out_dim=2, num_mixtures=9):
        super(MDN, self).__init__()
        in_dim = 3

        self.hidden = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, 128),
            # nn.ReLU(),
            # nn.Dropout(p=0.2),
            # nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.pi = nn.Linear(64, num_mixtures)
        self.mu = nn.Linear(64, num_mixtures * out_dim)
        self.sigma = nn.Linear(64, num_mixtures * out_dim)
        self.num_mixtures = num_mixtures
        self.out_dim = out_dim

    def forward(self, x_tuple):
        x, y, t = x_tuple
        concat = torch.cat([x, y, t], dim=1)
        h = self.hidden(concat)
        pi = torch.softmax(self.pi(h), dim=1)
        mu = self.mu(h).view(-1, self.num_mixtures, self.out_dim)
        sigma = torch.exp(self.sigma(h)).clamp(min=1e-2).view(-1, self.num_mixtures, self.out_dim)
        return pi, mu, sigma

# --------------------- MDN Loss ---------------------
def mdn_loss(pi, mu, sigma, y):
    y = y.unsqueeze(1).expand_as(mu)
    prob = (1.0 / (np.sqrt(2 * np.pi) * sigma)) * torch.exp(-0.5 * ((y - mu) / sigma) ** 2)
    prob = torch.prod(prob, dim=2)
    weighted = pi * prob
    log_prob = torch.log(torch.sum(weighted, dim=1) + 1e-6)
    return torch.mean(-log_prob)
    
# --------------------- Train ---------------------
def train_one_epoch(model, dataloader, optimizer, device, writer, global_sample_count, log_every=10000):
    model.train()
    total_loss = 0
    last_log_point = global_sample_count

    for x, y in tqdm(dataloader, desc="[Train]", leave=False):
        x = tuple(t.to(device) for t in x)
        y = y.to(device)
        batch_size = y.size(0)

        pi, mu, sigma = model(x)
        loss = mdn_loss(pi, mu, sigma, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        global_sample_count += batch_size

        if global_sample_count - last_log_point >= log_every:
            writer.add_scalar("Loss/Train_Per_10000", loss.item(), global_sample_count)
            last_log_point = global_sample_count

    avg_loss = total_loss / len(dataloader)
    print(f"[train_one_epoch] avg loss: {avg_loss:.6f}")
    return avg_loss, global_sample_count

def eval_one_epoch(model, dataloader, device, writer=None, epoch=None):
    model.eval()
    total_loss = 0
    visualized = False

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="[Eval ]", leave=False):
            x = tuple(t.to(device) for t in x)
            y = y.to(device)
            pi, mu, sigma = model(x)
            loss = mdn_loss(pi, mu, sigma, y)
            total_loss += loss.item()

            if not visualized and writer is not None and epoch is not None:
                fig = plt.figure(figsize=(12, 6))
                for i in range(min(8, y.size(0))):
                    ax = fig.add_subplot(2, 4, i + 1)
                    visualize_mdn_output(ax, model, tuple(t[i].unsqueeze(0) for t in x), device)
                fig.tight_layout()
                writer.add_figure("Eval/GMM_Contours", fig, global_step=epoch)
                plt.close(fig)
                visualized = True

    return total_loss / len(dataloader)

def train_loop(model, train_loader, eval_loader, optimizer, device,
               epochs=20, log_dir="runs", save_dir="saved_models", log_every=100000):
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(save_dir, exist_ok=True)
    writer = SummaryWriter(log_dir)

    global_sample_count = 0

    for epoch in range(1, epochs + 1):
        train_loss, global_sample_count = train_one_epoch(
            model, train_loader, optimizer, device, writer,
            global_sample_count, log_every=log_every
        )

        eval_loss = eval_one_epoch(model, eval_loader, device, writer=writer, epoch=epoch)
        writer.add_scalar("Loss/Eval_Epoch", eval_loss, epoch)

        print(f"[Epoch {epoch}] Train Loss: {train_loss:.6f} | Eval Loss: {eval_loss:.6f}")

        save_path = os.path.join(save_dir, f"mdn_epoch_{epoch:03d}.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'eval_loss': eval_loss,
        }, save_path)
        print(f"✅ Saved model to {save_path}")

    writer.close()

# --------------------- 메인 ---------------------
def main(batch_size=4096, log_every=10000, train_sample_ratio=1.0, eval_sample_ratio=1.0, seed=42, data_pattern="mdn_data_*.npz"):
    data_dir = "../../../Downloads/archive/processed_npy"
    log_dir = f"runs/mdn_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    save_dir = "saved_models"

    X, Y = load_dataset(data_dir, pattern=data_pattern)
    train_set, eval_set = split_dataset(
        X, Y,
        train_ratio=0.8,
        train_sample_ratio=train_sample_ratio,
        eval_sample_ratio=eval_sample_ratio,
        seed=seed
    )

    train_loader = get_dataloader(train_set, batch_size=batch_size)
    eval_loader = get_dataloader(eval_set, batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MDN(out_dim=2, num_mixtures=9).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

    train_loop(model, train_loader, eval_loader, optimizer, device,
               epochs=100, log_dir=log_dir, save_dir=save_dir, log_every=log_every)

In [3]:
# main(train_sample_ratio=1.0, eval_sample_ratio=1.0, data_pattern="mdn_data_far_*.npz")
main(train_sample_ratio=1.0, eval_sample_ratio=1.0, data_pattern="mdn_data_mid_*.npz")

[Train]:   0%|          | 0/13 [00:00<?, ?it/s]

[train_one_epoch] avg loss: 1.782183


[Eval ]:   0%|          | 0/4 [00:00<?, ?it/s]

[Epoch 1] Train Loss: 1.782183 | Eval Loss: 1.490877
✅ Saved model to saved_models\mdn_epoch_001.pt


[Train]:   0%|          | 0/13 [00:00<?, ?it/s]

...

[Epoch 99] Train Loss: 0.715162 | Eval Loss: 0.667766
✅ Saved model to saved_models\mdn_epoch_099.pt


[Train]:   0%|          | 0/13 [00:00<?, ?it/s]

[train_one_epoch] avg loss: 0.713604


[Eval ]:   0%|          | 0/4 [00:00<?, ?it/s]

[Epoch 100] Train Loss: 0.713604 | Eval Loss: 0.664661
✅ Saved model to saved_models\mdn_epoch_100.pt
